In [ ]:
import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import sys

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [ ]:
seed = 25
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
patience = 15
encoder_hidden_dim1 = 128
encoder_hidden_dim2 = 64
encoding_dim = 32
decoder_hidden_dim1 = 64
decoder_hidden_dim2 = 128
lr=0.001
n_splits = 5

In [ ]:
rng = np.random.RandomState(seed)

In [ ]:
data = pd.read_parquet("../data/GBG500.parquet")
data

In [ ]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

In [ ]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [ ]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

In [ ]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

In [ ]:
# Define Autoencoder architecture with Lightning
class Autoencoder(L.LightningModule):
    def __init__(
        self,
        input_dim,
        encoding_dim,
        encoder_hidden_dim1,
        encoder_hidden_dim2,
        decoder_hidden_dim1,
        decoder_hidden_dim2,
        lr,
    ):
        super().__init__()
        self.save_hyperparameters()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoder_hidden_dim1),
            nn.ReLU(),
            nn.Linear(encoder_hidden_dim1, encoder_hidden_dim2),
            nn.ReLU(),
            nn.Linear(encoder_hidden_dim2, encoding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, decoder_hidden_dim1),
            nn.ReLU(),
            nn.Linear(decoder_hidden_dim1, decoder_hidden_dim2),
            nn.ReLU(),
            nn.Linear(decoder_hidden_dim2, input_dim)
        )
        self.criterion = nn.MSELoss()
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        return self.encoder(x)
    
    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_hat = self.forward(x)
        loss = self.criterion(x_hat, x)
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
# 5-fold stratified CV with AE: collect out-of-fold latent representations
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
Z_oof = np.full((len(y_true), encoding_dim), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    fold_scaler = StandardScaler()
    X_train = fold_scaler.fit_transform(X_feat[train_idx])
    X_test = fold_scaler.transform(X_feat[test_idx])

    L.seed_everything(rng.randint(1000))
    autoencoder = Autoencoder(
        input_dim=X_train.shape[1],
        encoding_dim=encoding_dim,
        encoder_hidden_dim1=encoder_hidden_dim1,
        encoder_hidden_dim2=encoder_hidden_dim2,
        decoder_hidden_dim1=decoder_hidden_dim1,
        decoder_hidden_dim2=decoder_hidden_dim2,
        lr=lr,
    )

    X_train_tensor = torch.FloatTensor(X_train)
    dataset = TensorDataset(X_train_tensor, X_train_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    state = {'best_ewm_loss': float('inf'), 'best_state': None, 'ewm': None}

    class FoldLossTracker(L.pytorch.callbacks.Callback):
        def on_train_epoch_end(self, trainer, pl_module):
            loss = float(trainer.callback_metrics['train_loss'])
            if state['ewm'] is None:
                state['ewm'] = loss
            else:
                state['ewm'] = 0.2 * loss + 0.8 * state['ewm']
            if state['ewm'] < state['best_ewm_loss']:
                state['best_ewm_loss'] = state['ewm']
                state['best_state'] = copy.deepcopy(pl_module.state_dict())
            pl_module.log('train_loss_ewm_avg', state['ewm'])

    trainer = L.Trainer(
        max_epochs=300, accelerator='auto', devices=1,
        callbacks=[
            EarlyStopping(monitor='train_loss_ewm_avg', patience=patience,
                          verbose=False, mode='min', check_on_train_epoch_end=True),
            FoldLossTracker(),
        ],
        enable_progress_bar=False,
    )
    trainer.fit(autoencoder, dataloader)

    if state['best_state'] is not None:
        autoencoder.load_state_dict(state['best_state'])

    autoencoder.eval()
    X_test_tensor = torch.FloatTensor(X_test)
    with torch.no_grad():
        Z_train = autoencoder.encode(X_train_tensor).numpy()
        Z_test = autoencoder.encode(X_test_tensor).numpy()

    enc_scaler = StandardScaler()
    enc_scaler.fit(Z_train)
    Z_oof[test_idx] = enc_scaler.transform(Z_test)

    print(f"Fold {fold+1}/{n_splits} done")

In [ ]:
# Pick the two latent dimensions with the largest variance
variances = Z_oof.var(axis=0)
dim_x, dim_y = np.argsort(variances)[::-1][:2]
Z_2d = Z_oof[:, [dim_x, dim_y]]

category_styles = {
    'Safe':         dict(color='#cccccc', s=18, alpha=0.6, zorder=1),
    'Safe overall': dict(color='#2ca02c', s=28, alpha=0.8, zorder=2),
    'Bad weather':  dict(color='#4c72b0', s=28, alpha=0.8, zorder=2),
    'Tandem':       dict(color='#8172b3', s=28, alpha=0.8, zorder=2),
    'Reckless':     dict(color='#c44e52', s=90, alpha=1.0, zorder=3,
                         marker='*', edgecolors='black', linewidths=0.6),
}

fig, ax = plt.subplots(figsize=(9, 7))
for category, style in category_styles.items():
    mask = labels_sorted['label'].values == category
    ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1], label=category, **style)

ax.set_xlabel("latent dim (highest variance)", fontsize=16)
ax.set_ylabel("latent dim (second highest variance)", fontsize=16)
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc='best', framealpha=0.9, fontsize=14)
plt.tight_layout()

fig.savefig(f"../plots/GBG500_latent_space.png", dpi=300, bbox_inches='tight')
plt.show()